### Bronze Layer Data Ingestion

In [0]:
from pyspark.sql import functions as F
import os

source_path = "/Workspace/Users/dineshdhina300@gmail.com/Aerospace_prjct/Aerospace_source_files"

# Automatically list all files in the directory and filter for CSVs
files = [f.name for f in dbutils.fs.ls(f"file:{source_path}") if f.name.endswith(".csv")]

for file in files:
    file_path = f"{source_path}/{file}"
    
    # Read CSV
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(file_path)
    )
    
    # Add ingestion timestamp
    df = df.withColumn(
        "ingestion_timestamp", F.current_timestamp()
    )
    
    # Create table name
    table_name = file.replace(".csv", "").lower()
    
    # Write to Unity Catalog Bronze table
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"workspace.default.bronze_{table_name}")
    )
    
    print(f"Loaded: {file} -> bronze_{table_name}")